# INTERSECTON

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
import warnings
warnings.filterwarnings('ignore')

from scipy.linalg import sqrtm, cholesky, cho_solve
import nonlinshrink as nls

def nlshrink_cov(Y, k=1):
    """
    Nonlinear shrinkage covariance estimation.
    
    Parameters:
    -----------
    Y : array-like, shape (n_samples, n_features)
        Data matrix
    k : int
        Number of factors (parameter for compatibility, may not be used)
    
    Returns:
    --------
    theta : array, shape (n_features, n_features)
        Shrinkage covariance estimate
    """
    # Using the non-linear-shrinkage package
    # This implements the Ledoit & Wolf nonlinear shrinkage method
    return nls.shrink_cov(Y)


def msr_weights(Theta_hat, mu):
    """
    Compute Maximum Sharpe Ratio portfolio weights.
    
    The maximum Sharpe ratio portfolio solves:
    max (w' mu) / sqrt(w' Sigma w)
    
    Solution (when mu represents excess returns):
    w ∝ Sigma^{-1} mu = Theta mu
    
    Then normalize so that sum(w) = 1.
    
    Parameters:
    -----------
    Theta_hat : np.ndarray, shape (p, p)
        Precision matrix (Sigma^{-1})
    mu : np.ndarray, shape (p,)
        Expected excess returns
    
    Returns:
    --------
    w_star : np.ndarray, shape (p,)
        Portfolio weights (sum to 1)
    """
    p = Theta_hat.shape[0]
    ones_p = np.ones(p)
    
    # Compute unnormalized weights: w ∝ Theta mu
    w_unnorm = Theta_hat @ mu
    
    # Normalize to sum to 1
    weight_sum = np.sum(w_unnorm)
    
    if np.abs(weight_sum) < 1e-10:
        print('WARNING: Weight sum near zero, returning equal weights')
        return ones_p / p
    
    w_star = w_unnorm / weight_sum
    
    return w_star


def load_yearly_signals(year, buys_path_template='buys_{}.csv', sells_path_template='sells_{}.csv'):
    """
    Load buy and sell signals for a specific year.
    
    Parameters:
    -----------
    year : int
        Year to load signals for
    buys_path_template : str
        Template for buys file path (use {} for year placeholder)
    sells_path_template : str
        Template for sells file path (use {} for year placeholder)
    
    Returns:
    --------
    permno_set : set
        Set of permnos in the buy and sell signals for this year
    """
    try:
        buys = pd.read_csv(buys_path_template.format(year), index_col=1)
        sells = pd.read_csv(sells_path_template.format(year), index_col=1)
        
        buys.index.name = 'permno'
        sells.index.name = 'permno'
        
        buys_index = buys.index.astype(int)
        sells_index = sells.index.astype(int)
        
        return set(buys_index.union(sells_index))
    except FileNotFoundError as e:
        print(f"  ⚠ Warning: Could not load signals for year {year}: {e}")
        return set()


def load_finbert_signals(signals_path):
    """
    Load FinBERT monthly signals from CSV file.
    
    Parameters:
    -----------
    signals_path : str
        Path to monthly_signals.csv file
    
    Returns:
    --------
    signals_df : pd.DataFrame
        DataFrame with columns: symbol, company, year_month, signal, avg_sentiment_score
    """
    try:
        signals_df = pd.read_csv(signals_path)
        # Convert year_month to datetime (end of month)
        signals_df['date'] = pd.to_datetime(signals_df['year_month']) + pd.offsets.MonthEnd(0)
        return signals_df
    except FileNotFoundError as e:
        print(f"  ⚠ Warning: Could not load FinBERT signals: {e}")
        return pd.DataFrame(columns=['symbol', 'company', 'year_month', 'signal', 'date'])


def get_finbert_permnos_for_date(signals_df, ticker_to_permno, date):
    """
    Get set of permnos with 'buy' or 'sell' signals for a specific date.
    
    Parameters:
    -----------
    signals_df : pd.DataFrame
        FinBERT signals dataframe
    ticker_to_permno : dict
        Mapping from ticker symbol to permno
    date : pd.Timestamp
        Date to get signals for
    
    Returns:
    --------
    permno_set : set
        Set of permnos with buy or sell signals on this date
    """
    # Get signals for this date
    date_signals = signals_df[signals_df['date'] == date]
    
    # Filter for buy and sell signals (exclude hold)
    buy_signals = date_signals[date_signals['signal'] == 'buy']
    sell_signals = date_signals[date_signals['signal'] == 'sell']
    
    # Convert tickers to permnos
    permnos = set()
    for ticker in buy_signals['symbol'].values:
        if ticker in ticker_to_permno:
            permnos.add(ticker_to_permno[ticker])
    for ticker in sell_signals['symbol'].values:
        if ticker in ticker_to_permno:
            permnos.add(ticker_to_permno[ticker])
    
    return permnos


def create_ticker_to_permno_mapping(df):
    """
    Create a mapping from ticker to permno from the returns dataframe.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Returns dataframe with 'ticker' and 'permno' columns
    
    Returns:
    --------
    ticker_to_permno : dict
        Mapping from ticker to permno (uses most recent permno for each ticker)
    """
    if 'ticker' not in df.columns:
        raise ValueError("DataFrame must have 'ticker' column for mapping")
    
    # Drop NaN tickers
    valid_df = df[df['ticker'].notna()].copy()
    
    # Get the most recent permno for each ticker
    ticker_to_permno = valid_df.groupby('ticker')['permno'].last().to_dict()
    
    return ticker_to_permno


def calculate_exit_transaction_cost(prev_weights_dict, prev_oos_returns_dict, 
                                    prev_gross_return, transaction_cost, verbose=False):
    """
    Calculate transaction cost when exiting the market (liquidating all positions).
    Enforces Immediate Liquidation logic:
    - Next period return is 0.0 (Cash)
    - Cost is paid on current portfolio value
    """
    if len(prev_weights_dict) == 0:
        return 0.0, 0.0, 0.0
    
    # Step 1: Adjust previous weights to current period's BEGINNING (drift from t-1 to t)
    adjusted_prev = {}
    for asset, prev_w in prev_weights_dict.items():
        if asset in prev_oos_returns_dict:
            prev_r = prev_oos_returns_dict[asset]
            if abs(1 + prev_gross_return) > 1e-6:
                adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_return)
            else:
                adjusted_prev[asset] = 0.0
        else:
            if abs(1 + prev_gross_return) > 1e-6:
                adjusted_prev[asset] = prev_w / (1 + prev_gross_return)
            else:
                adjusted_prev[asset] = 0.0
    
    # Step 2: Turnover (Selling everything to Cash)
    turnover = sum(abs(w) for w in adjusted_prev.values())
    
    # Step 3: Cost 
    # Paper Formula: c * (1 + R_next) * Turnover. 
    # Since R_next (Cash) is 0.0, this simplifies to c * 1.0 * Turnover.
    tc = transaction_cost * 1.0 * turnover
    
    # Step 4: Net Return is 0.0 (Cash return) - Cost
    net_return = -tc
    
    if verbose:
        print(f"  Liquidating positions | Turnover: {turnover:>6.4f} | TC: {tc:>8.6f}")
    
    return turnover, tc, net_return

def backtest_nodewise_gmv_combined(df, 
                                    test_start_date='2020-01-31', 
                                    test_end_date='2024-11-30',
                                    lookback_window=180,
                                    transaction_cost=0.001,
                                    buys_path_template='buys_{}.csv',
                                    sells_path_template='sells_{}.csv',
                                    finbert_signals_path=None,
                                    verbose=True):
    """
    Backtest Nodewise + GMV strategy with year-specific buy/sell signals 
    and FinBERT sentiment signals.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with columns: permno, datadate, ticker, ret_fwd_1
    test_start_date : str
        First date for out-of-sample returns (format: 'YYYY-MM-DD')
    test_end_date : str
        Last date for out-of-sample returns (format: 'YYYY-MM-DD')
    lookback_window : int
        Number of months in rolling training window (default: 180)
    transaction_cost : float
        Proportional transaction cost (default: 0.001 = 10 bps)
    buys_path_template : str
        Template for buys file path (use {} for year placeholder)
    sells_path_template : str
        Template for sells file path (use {} for year placeholder)
    finbert_signals_path : str or None
        Path to FinBERT signals CSV file. If None, only uses buy/sell signals.
    verbose : bool
        If True, prints detailed log at each time step.
    
    Returns:
    --------
    results_df : pd.DataFrame
        DataFrame with columns: date, portfolio_return, cumulative_return
    metrics : dict
        Overall performance metrics
    """
    # --- 1. Setup ---
    df = df.copy()
    if 'datadate' not in df.columns or 'permno' not in df.columns:
        raise ValueError("DataFrame must have 'datadate' and 'permno' columns")
    df['datadate'] = pd.to_datetime(df['datadate'])
    
    # Create ticker to permno mapping
    if verbose:
        print("Creating ticker to permno mapping...")
    ticker_to_permno = create_ticker_to_permno_mapping(df)
    if verbose:
        print(f"Mapped {len(ticker_to_permno)} unique tickers to permnos")
    
    # Load FinBERT signals if provided
    finbert_df = None
    if finbert_signals_path is not None:
        finbert_df = load_finbert_signals(finbert_signals_path)
        if verbose and len(finbert_df) > 0:
            print(f"Loaded FinBERT signals: {len(finbert_df)} monthly records")
            print(f"FinBERT signal distribution:")
            print(finbert_df['signal'].value_counts())
    
    # Get unique dates
    all_dates = sorted(df['datadate'].unique())
    
    # Convert test dates to datetime
    test_start_dt = pd.to_datetime(test_start_date)
    test_end_dt = pd.to_datetime(test_end_date)
    
    # Find date indices
    try:
        test_start_idx = all_dates.index(test_start_dt)
        test_end_idx = all_dates.index(test_end_dt)
    except ValueError as e:
        raise ValueError(f"Date not found in DataFrame: {e}")
    
    if test_start_idx < lookback_window:
        raise ValueError(f"Not enough data for lookback. Test start date {test_start_date} "
                         f"requires data back to {all_dates[test_start_idx - lookback_window]}, "
                         f"but only {test_start_idx} periods are available.")
    
    # Storage for results
    portfolio_returns = []
    portfolio_dates = []
    portfolio_weights_list = []
    portfolio_turnover_list = []
    portfolio_gross_returns = []
    
    # Track weights by permno
    prev_weights_dict = {}
    prev_oos_returns_dict = {}
    prev_gross_return = 0.0
    
    # Cache for yearly signals
    yearly_signals_cache = {}
    
    # --- 2. Rolling Window Backtest ---
    if verbose:
        print("="*60)
        print("STARTING BACKTEST WITH YEARLY + FINBERT SIGNALS")
        print("="*60)
        
    for t in range(test_start_idx, test_end_idx + 1):
        current_date = all_dates[t]
        current_year = current_date.year
        
        # Load signals for current year if not cached
        if current_year not in yearly_signals_cache:
            yearly_signals_cache[current_year] = load_yearly_signals(
                current_year, buys_path_template, sells_path_template
            )
        
        yearly_permnos = yearly_signals_cache[current_year]
        
        # Get FinBERT signals for current date
        finbert_permnos = set()
        if finbert_df is not None and len(finbert_df) > 0:
            finbert_permnos = get_finbert_permnos_for_date(finbert_df, ticker_to_permno, current_date)
        
        # UNION of yearly signals and FinBERT signals
        allowed_permnos = yearly_permnos.intersection(finbert_permnos)
        if len(allowed_permnos) <= 1:
            allowed_permnos = yearly_permnos.union(finbert_permnos)
        
        # ========================================
        # CRITICAL: Get OOS returns FIRST before any early exits
        # ========================================
        oos_data = df[(df['datadate'] == current_date) & (df['permno'].isin(allowed_permnos))]
        oos_returns_series = oos_data.set_index('permno')['ret_fwd_1']
        oos_returns_series = oos_returns_series.dropna()
        oos_returns_dict = oos_returns_series.to_dict()
        
        # Now handle early exit cases
        if len(allowed_permnos) == 0:
            if verbose:
                print(f"\n[{t - test_start_idx + 1}/{test_end_idx - test_start_idx + 1}] "
                      f"Date: {current_date.strftime('%Y-%m-%d')}")
                print(f"  ⚠ No signals for {current_date.strftime('%Y-%m-%d')}, recording zero return")
            
            # Now oos_returns_dict is defined and can be passed
            turnover, tc, net_return = calculate_exit_transaction_cost(
                prev_weights_dict, 
                prev_oos_returns_dict, 
                prev_gross_return, 
                transaction_cost,
                verbose=verbose
            )
            
            portfolio_returns.append(net_return)
            portfolio_dates.append(current_date)
            portfolio_weights_list.append({})
            portfolio_turnover_list.append(turnover)
            portfolio_gross_returns.append(0.0)
            
            prev_weights_dict = {}
            prev_oos_returns_dict = {}
            prev_gross_return = 0.0
            continue
        
        # Define the lookback window
        window_start_date = all_dates[t - lookback_window]
        window_end_date = all_dates[t - 1]
        
        # Get training data for this window
        train_data = df[(df['datadate'] >= window_start_date) & 
                        (df['datadate'] <= window_end_date) &
                        (df['permno'].isin(allowed_permnos))]
        
        # Pivot to get returns matrix
        returns_pivot = train_data.pivot(index='datadate', columns='permno', values='ret_fwd_1')
        window_dates = all_dates[t - lookback_window : t]
        returns_pivot = returns_pivot.reindex(index=window_dates)
        
        # Filter assets with any NaNs
        nan_assets = returns_pivot.columns[returns_pivot.isna().any()]
        filtered_pivot = returns_pivot.drop(columns=nan_assets)
        
        current_assets = filtered_pivot.columns.tolist()
        Y = filtered_pivot.values
        n_train, p_current = Y.shape
    
        if verbose:
            print(f"\n[{t - test_start_idx + 1}/{test_end_idx - test_start_idx + 1}] "
                  f"Date: {current_date.strftime('%Y-%m-%d')} | Year: {current_year}")
            print(f"  Window: {window_start_date.strftime('%Y-%m-%d')} to "
                  f"{window_end_date.strftime('%Y-%m-%d')}")
            print(f"  Yearly: {len(yearly_permnos)} | FinBERT: {len(finbert_permnos)} | "
                  f"Union/Intersection: {len(allowed_permnos)} | Assets w/ data: {p_current}")
    
        # Check for valid data
        if n_train < lookback_window or p_current < 2:
            if verbose:
                print(f"  ⚠ Insufficient data (n={n_train}, p={p_current}), recording zero return")
            
            turnover, tc, net_return = calculate_exit_transaction_cost(
                prev_weights_dict, 
                prev_oos_returns_dict, 
                prev_gross_return, 
                transaction_cost,
                verbose=verbose
            )
            
            portfolio_returns.append(net_return)
            portfolio_dates.append(current_date)
            portfolio_weights_list.append({})
            portfolio_turnover_list.append(turnover)
            portfolio_gross_returns.append(0.0)
            
            prev_weights_dict = {}
            prev_oos_returns_dict = {}
            prev_gross_return = 0.0
            continue
        
        try:
            # Demean the returns
            Y_bar = Y.mean(axis=0)
            Y_star = Y - Y_bar
            
            if verbose:
                print(f"  Running NLS Regression...")
            Sigma_hat = nlshrink_cov(Y_star)
            # Invert to get precision matrix
            try:
                Theta_hat = np.linalg.inv(Sigma_hat)
            except np.linalg.LinAlgError:
                # If inversion fails, use pseudo-inverse
                if verbose:
                    print(f"  ⚠ Covariance matrix singular, using pseudo-inverse")
                Theta_hat = np.linalg.pinv(Sigma_hat)
            
            if verbose:
                print(f"  Computing MSR weights...")

            mu = Y.mean(axis=0)
            
            w_star = msr_weights(Theta_hat, mu)
            
            # Create weights dictionary
            new_weights_dict = {asset: w_star[i] for i, asset in enumerate(current_assets)}
            
        except Exception as e:
            if verbose:
                print(f"  ✗ Error: {e}")
                print(f"  Recording zero return")
            
            # Calculate transaction cost from liquidating previous positions
            turnover, tc, net_return = calculate_exit_transaction_cost(
            prev_weights_dict, 
            prev_oos_returns_dict, 
            prev_gross_return, 
            transaction_cost,
            verbose=verbose
        )
            
            # Record zero return with empty weights
            portfolio_returns.append(net_return)
            portfolio_dates.append(current_date)
            portfolio_weights_list.append({})
            portfolio_turnover_list.append(turnover)
            portfolio_gross_returns.append(0.0)
            
            # Reset previous state
            prev_weights_dict = {}
            prev_oos_returns_dict = {}
            prev_gross_return = 0.0
            continue

        # Normalize weights to sum to 1
        weight_sum = sum(new_weights_dict.values())
        if weight_sum > 1e-10:
            new_weights_dict = {k: v/weight_sum for k, v in new_weights_dict.items()}
        else:
            if verbose:
                print("  ⚠ Zero weight sum, recording zero return")
            
            # Calculate transaction cost from liquidating previous positions
            turnover, tc, net_return = calculate_exit_transaction_cost(
            prev_weights_dict, 
            prev_oos_returns_dict, 
            prev_gross_return, 
            transaction_cost,
            verbose=verbose
        )
            
            # Record zero return with empty weights
            portfolio_returns.append(net_return)
            portfolio_dates.append(current_date)
            portfolio_weights_list.append({})
            portfolio_turnover_list.append(turnover)
            portfolio_gross_returns.append(0.0)
            
            # Reset previous state
            prev_weights_dict = {}
            prev_oos_returns_dict = {}
            prev_gross_return = 0.0
            continue
        
        # --- 3. OOS Returns & Transaction Costs ---
        
        # Find common assets between weights and returns
        common_assets = set(new_weights_dict.keys()) & set(oos_returns_dict.keys())
        
        if len(common_assets) == 0:
            if verbose:
                print("  ⚠ No common assets with valid returns, recording zero return")
            
            # Calculate transaction cost from liquidating previous positions
            turnover, tc, net_return = calculate_exit_transaction_cost(
            prev_weights_dict, 
            prev_oos_returns_dict, 
            prev_gross_return, 
            transaction_cost,
            verbose=verbose
        )
            
            # Record zero return with empty weights
            portfolio_returns.append(net_return)
            portfolio_dates.append(current_date)
            portfolio_weights_list.append({})
            portfolio_turnover_list.append(turnover)
            portfolio_gross_returns.append(0.0)
            
            # Reset previous state
            prev_weights_dict = {}
            prev_oos_returns_dict = {}
            prev_gross_return = 0.0
            continue
        
        # Filter to common assets and renormalize
        common_weights = {a: new_weights_dict[a] for a in common_assets}
        common_weight_sum = sum(common_weights.values())
        if common_weight_sum > 1e-10:
            common_weights = {k: v/common_weight_sum for k, v in common_weights.items()}
        else:
            if verbose:
                print("  ⚠ Zero weight sum after filtering, recording zero return")
            
            # Calculate transaction cost from liquidating previous positions
            turnover, tc, net_return = calculate_exit_transaction_cost(
            prev_weights_dict, 
            prev_oos_returns_dict, 
            prev_gross_return, 
            transaction_cost,
            verbose=verbose
        )
            
            # Record zero return with empty weights
            portfolio_returns.append(net_return)
            portfolio_dates.append(current_date)
            portfolio_weights_list.append({})
            portfolio_turnover_list.append(turnover)
            portfolio_gross_returns.append(0.0)
            
            # Reset previous state
            prev_weights_dict = {}
            prev_oos_returns_dict = {}
            prev_gross_return = 0.0
            continue
        
        # Compute gross portfolio return
        gross_return = sum(common_weights[a] * oos_returns_dict[a] for a in common_assets)
        
        # Sanity check
        if np.isnan(gross_return) or np.isinf(gross_return):
            if verbose:
                print(f"  ⚠ Invalid gross return: {gross_return}, recording zero return")
            
            # Calculate transaction cost from liquidating previous positions
            turnover, tc, net_return = calculate_exit_transaction_cost(
            prev_weights_dict, 
            prev_oos_returns_dict, 
            prev_gross_return, 
            transaction_cost,
            verbose=verbose
        )
            
            # Record zero return with empty weights
            portfolio_returns.append(net_return)
            portfolio_dates.append(current_date)
            portfolio_weights_list.append({})
            portfolio_turnover_list.append(turnover)
            portfolio_gross_returns.append(0.0)
            
            # Reset previous state
            prev_weights_dict = {}
            prev_oos_returns_dict = {}
            prev_gross_return = 0.0
            continue
        
        # Calculate transaction costs
        if len(prev_weights_dict) > 0:
            # Step 1: Adjust ALL previous weights for returns
            # This gives us the portfolio composition after returns but before rebalancing
            adjusted_prev = {}
            
            for asset, prev_w in prev_weights_dict.items():
                if asset in prev_oos_returns_dict:
                    prev_r = prev_oos_returns_dict[asset]
                    # Safer division check
                    if abs(1 + prev_gross_return) > 1e-6:
                        adjusted_prev[asset] = prev_w * (1 + prev_r) / (1 + prev_gross_return)
                    else:
                        # Portfolio nearly went to zero - conservative approach
                        adjusted_prev[asset] = 0.0
                else:
                    print('ASLKDJHAFLKJSDFH')
                    # Asset had weight but no return data
                    # Conservative: assume it returned 0% (stayed at same value)
                    # So its weight adjusts by the inverse of portfolio growth
                    if abs(1 + prev_gross_return) > 1e-6:
                        adjusted_prev[asset] = prev_w / (1 + prev_gross_return)
                    else:
                        adjusted_prev[asset] = 0.0
            
            # Step 2: Calculate turnover
            # Get all assets involved in trading (held before OR after)
            all_assets = set(adjusted_prev.keys()) | set(common_weights.keys())
            
            # Turnover = sum of |new_weight - old_adjusted_weight| for all assets
            turnover = 0.0
            for asset in all_assets:
                old_w = adjusted_prev.get(asset, 0.0)  # Weight after returns, before rebalancing
                new_w = common_weights.get(asset, 0.0)  # Target weight after rebalancing
                turnover += abs(new_w - old_w)
            
            # Transaction cost on end-of-period portfolio value
            tc = transaction_cost * (1 + gross_return) * turnover
        
        else:
            # First period: no previous positions, buying into everything
            turnover = sum(abs(w) for w in common_weights.values())
            tc = transaction_cost * (1 + gross_return) * turnover
        
        # Net return
        net_return = gross_return - tc
        
        # Store results
        portfolio_returns.append(net_return)
        portfolio_dates.append(current_date)
        portfolio_weights_list.append(common_weights.copy())
        portfolio_turnover_list.append(turnover)
        portfolio_gross_returns.append(gross_return)
        
        # Update previous values for next iteration
        prev_weights_dict = common_weights.copy()
        prev_oos_returns_dict = {a: oos_returns_dict[a] for a in common_assets}
        prev_gross_return = gross_return
        
        if verbose:
            print(f"  Gross: {gross_return:>8.5f} | Turnover: {turnover:>6.4f} | "
                  f"TC: {tc:>8.6f} | Net: {net_return:>8.5f}")

    if verbose:
        print("\n" + "="*60)
        print("BACKTEST COMPLETE")
        print("="*60)
    
    # --- 4. Compile Results ---
    results_df = pd.DataFrame({
        'date': portfolio_dates,
        'portfolio_return': portfolio_returns,
        'portfolio_gross_return': portfolio_gross_returns,
        'portfolio_weights': portfolio_weights_list,
        'portfolio_turnover': portfolio_turnover_list
    })
    results_df['cumulative_return'] = (1 + results_df['portfolio_return']).cumprod() - 1
    
    # Compute overall metrics
    if len(portfolio_returns) > 0:
        mean_return = np.mean(portfolio_returns)
        variance = np.var(portfolio_returns, ddof=1)
        sharpe_ratio = mean_return / np.sqrt(variance) if variance > 0 else 0
        
        # Annualized metrics (monthly data)
        annual_return = mean_return * 12
        annual_volatility = np.sqrt(variance * 12)
        annual_sharpe = annual_return / annual_volatility if annual_volatility > 0 else 0
        
        metrics = {
            'mean_return': mean_return,
            'variance': variance,
            'sharpe_ratio': sharpe_ratio,
            'annual_return': annual_return,
            'annual_volatility': annual_volatility,
            'annual_sharpe_ratio': annual_sharpe,
            'total_return': results_df['cumulative_return'].iloc[-1],
            'avg_turnover': np.mean(portfolio_turnover_list),
            'n_periods': len(portfolio_returns),
            'n_zero_periods': sum(1 for r in portfolio_returns if r == 0)
        }
    else:
        metrics = {
            'mean_return': 0,
            'variance': 0,
            'sharpe_ratio': 0,
            'annual_return': 0,
            'annual_volatility': 0,
            'annual_sharpe_ratio': 0,
            'total_return': 0,
            'avg_turnover': 0,
            'n_periods': 0,
            'n_zero_periods': 0
        }
    
    return results_df, metrics

In [ ]:
df = pd.read_csv('../../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)

In [4]:
results_df, metrics = backtest_nodewise_gmv_combined(
    df,
    test_start_date='2023-11-30',
    test_end_date='2024-04-30',
    lookback_window=180,
    transaction_cost=0.001,
    buys_path_template='../buys_{}.csv',
    sells_path_template='../sells_{}.csv',
    finbert_signals_path='../../examples/monthly_signals_decay.csv',  # Your FinBERT signals
    verbose=True
)

print(f"\nSharpe Ratio: {metrics['sharpe_ratio']:.4f}")
print(f"Annualized Sharpe Ratio: {metrics['annual_sharpe_ratio']:.4f}")
print(f"Total Return: {metrics['total_return']:.4f}")
print(f"Average Turnover: {metrics['avg_turnover']:.4f}")

Creating ticker to permno mapping...
Mapped 1664 unique tickers to permnos
Loaded FinBERT signals: 24780 monthly records
FinBERT signal distribution:
signal
hold    23840
sell      529
buy       411
Name: count, dtype: int64
STARTING BACKTEST WITH YEARLY + FINBERT SIGNALS

[1/6] Date: 2023-11-30 | Year: 2023
  Window: 2008-11-30 to 2023-10-31
  Yearly: 126 | FinBERT: 35 | Union/Intersection: 9 | Assets w/ data: 8
  Running NLS Regression...
  Computing MSR weights...
  Gross:  0.04337 | Turnover: 1.4082 | TC: 0.001469 | Net:  0.04190

[2/6] Date: 2023-12-31 | Year: 2023
  Window: 2008-12-31 to 2023-11-30
  Yearly: 126 | FinBERT: 36 | Union/Intersection: 13 | Assets w/ data: 13
  Running NLS Regression...
  Computing MSR weights...
  Gross:  0.05975 | Turnover: 2.4554 | TC: 0.002602 | Net:  0.05714

[3/6] Date: 2024-01-31 | Year: 2024
  Window: 2009-01-31 to 2023-12-31
  Yearly: 6 | FinBERT: 49 | Union/Intersection: 55 | Assets w/ data: 29
  Running NLS Regression...
  Computing MSR wei

In [5]:
metrics['mean_return']*12

0.5262786563046493

In [6]:
metrics['variance']*12

0.0041583411480356144

In [7]:
metrics['avg_turnover']

2.891410928135853

In [25]:
import pandas as pd
import numpy as np

# Load the permno-comnam mapping
permno_comnam = pd.read_csv('../../permno-comnam.csv')

# Create a dictionary mapping permno to comnam for quick lookup
permno_to_comnam = dict(zip(permno_comnam['permno'].astype(str), 
                             permno_comnam['comnam']))


def add_company_names(weights_dict):
    """
    Takes a dictionary from portfolio_weights column,
    adds company names to permnos, and returns the updated dictionary.
    """
    # Handle empty dictionaries or NaN values
    if not weights_dict or (isinstance(weights_dict, float) and pd.isna(weights_dict)):
        return {}
    
    # weights_dict is already a dictionary, no need to parse
    # Create new dictionary with "permno: comnam" as keys
    updated_dict = {}
    for permno, weight in weights_dict.items():
        permno_str = str(permno)
        comnam = permno_to_comnam.get(permno_str, 'Unknown')
        new_key = f"{permno}: {comnam}"
        updated_dict[new_key] = weight
    
    return updated_dict

# Apply the function to portfolio_weights column
results_df['portfolio_weights_with_names'] = results_df['portfolio_weights'].apply(add_company_names)

# Set pandas display options to show full content
pd.set_option('display.max_colwidth', None)  # No limit on column width
pd.set_option('display.max_rows', None)      # Show all rows (optional)
pd.set_option('display.width', None)         # Auto-detect width

# Display first few rows to verify
#print(results_df[['date', 'portfolio_weights_with_names']])

# --- Diagnostics: print holdings and one-month-ahead returns per date ---
print('\n--- Diagnostics: holdings and one-month-ahead returns per period ---')
for _, row in results_df.iterrows():
    date = row['date']
    weights = row['portfolio_weights'] if isinstance(row['portfolio_weights'], dict) else {}
    if weights:
        print(f"\nDate: {pd.to_datetime(date).strftime('%Y-%m-%d')}")
        for permno in sorted(weights.keys()):
            weight = weights[permno]
            mask = (pd.to_datetime(df['datadate']) == pd.to_datetime(date)) & (df['permno'] == permno)
            matched = df.loc[mask, 'ret_fwd_1']
            r = matched.iloc[0] if not matched.empty else float('nan')
            comnam = permno_to_comnam.get(str(permno), 'Unknown')
            try:
                print(f"  {permno}: {comnam} | weight={weight:.6f} | ret_fwd_1={r:.6f}")
            except Exception:
                print(f"  {permno}: {comnam} | weight={weight} | ret_fwd_1={r}")

# --- Summary statistics to help diagnose Sharpe ---
print('\n--- Summary Statistics ---')
rets = results_df['portfolio_return'].dropna()
mean = rets.mean() if len(rets)>0 else float('nan')
std = rets.std(ddof=1) if len(rets)>1 else float('nan')
print(f"Periods: {len(rets)}")
print(f"Mean (monthly): {mean:.6f}")
print(f"Std (monthly): {std:.6f}")
print(f"Computed Sharpe (mean/std, monthly): {(mean/std) if std and std>0 else 'nan'}")
if std and std>0:
    annual_sharpe = (mean * 12) / (std * np.sqrt(12))
    print(f"Annualized Sharpe (mean*12 / (std*sqrt(12))): {annual_sharpe:.4f}")

# Asset concentration diagnostics
all_assets = set()
holdings_counts = []
for w in results_df['portfolio_weights']:
    if isinstance(w, dict) and w:
        all_assets.update(w.keys())
        holdings_counts.append(len([x for x in w.values() if abs(x)>1e-8]))

print(f"\nUnique assets across periods: {len(all_assets)}")
print(f"Avg number of holdings per period: {np.mean(holdings_counts) if holdings_counts else 0}")

# Print list of unique assets and their average weights/returns
# if all_assets:
#     print('\nAsset-level averages:')
#     for permno in sorted(all_assets):
#         # average weight across periods
#         weights_list = [ (w.get(permno,0) if isinstance(w,dict) else 0) for w in results_df['portfolio_weights']]
#         avg_w = np.mean([x for x in weights_list if abs(x)>0]) if any(abs(x)>0 for x in weights_list) else 0
#         # average realized ret_fwd_1 when held
#         rets_when_held = []
#         for _, row in results_df.iterrows():
#             if isinstance(row['portfolio_weights'], dict) and permno in row['portfolio_weights']:
#                 date = row['date']
#                 mask = (pd.to_datetime(df['datadate']) == pd.to_datetime(date)) & (df['permno'] == permno)
#                 matched = df.loc[mask, 'ret_fwd_1']
#                 if not matched.empty:
#                     rets_when_held.append(matched.iloc[0])
#         avg_ret = np.mean(rets_when_held) if len(rets_when_held)>0 else float('nan')
#         comnam = permno_to_comnam.get(str(permno), 'Unknown')
#         print(f"  {permno}: {comnam} | avg_weight_when_held={avg_w:.6f} | avg_ret_when_held={avg_ret}")



--- Diagnostics: holdings and one-month-ahead returns per period ---

Date: 2023-11-30
  25081: COMERICA INC | weight=0.095833 | ret_fwd_1=0.245589
  26403: DISNEY WALT CO | weight=-0.103099 | ret_fwd_1=-0.026956
  43449: MCDONALDS CORP | weight=0.467541 | ret_fwd_1=0.047751
  55976: WALMART INC | weight=0.217614 | ret_fwd_1=0.011950
  64995: KEYCORP NEW | weight=-0.059677 | ret_fwd_1=0.157928
  75510: ADOBE INC | weight=0.210081 | ret_fwd_1=-0.027884
  76226: PARAMOUNT GLOBAL | weight=-0.041331 | ret_fwd_1=0.028407
  90215: SALESFORCE INC | weight=0.213038 | ret_fwd_1=0.040321

Date: 2023-12-31
  11308: COCA COLA CO | weight=0.051581 | ret_fwd_1=0.004803
  19393: BRISTOL MYERS SQUIBB CO | weight=-0.036350 | ret_fwd_1=-0.040561
  24205: NEXTERA ENERGY INC | weight=0.182045 | ret_fwd_1=-0.039438
  25582: L3HARRIS TECHNOLOGIES INC | weight=0.124930 | ret_fwd_1=-0.015145
  35554: M & T BANK CORP | weight=0.076390 | ret_fwd_1=0.002741
  50876: LILLY ELI & CO | weight=0.350001 | ret_fwd_1=